# Prompt and Answer Prototype

This notebook is for validating prompt assembly and, when Ollama is running locally, end-to-end answer generation before relying on the Discord bot flow.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rag.llm import build_prompt
from rag.retriever import retrieve


c:\DiscordBot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
history = [
    ("How do refunds work?", "Refunds depend on the subscription type and eligibility rules."),
]
chunks = retrieve(
    query="Can annual customers get a refund?",
    db_path=str(PROJECT_ROOT / "data" / "vectors.db"),
    top_k=3,
)
prompt = build_prompt("Can annual customers get a refund?", chunks, history)

prompt[:2000]


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3712.92it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


"System Context:\nYou are TechNova's support assistant. Answer using the retrieved knowledge base context when it is relevant. If the context does not contain the answer, say that the information is not available in the knowledge base. Be concise, accurate, and do not invent policies, pricing, or product details.\n\nRetrieved Chunks:\n[Source 1: return_policy.md]\n# TechNova Return and Refund Policy ## Policy Summary This policy explains how TechNova handles cancellations, refunds, and service-related billing disputes for subscriptions and add-on services. Because TechNova is a software platform, there are no physical product returns. Instead, requests are reviewed based on subscription status, usage, and the type of purchase made. ## Monthly Subscriptions Customers on monthly plans may cancel at any time from the billing settings page. Cancellation stops future renewal charges, but the current billing period is not prorated or refunded once it has started. Access remains available unt

In [3]:
from rag.llm import generate_answer
generate_answer("Can annual customers get a refund?", chunks, history)

{'answer': 'According to our Return and Refund Policy (Source 1), annual subscriptions are billed in advance and are intended for long-term use. Unfortunately, this means that we do not provide partial refunds for unused months on annual contracts.\n\nHowever, new annual customers may request a full refund within 14 calendar days of the original purchase date if they have used fewer than 500 workflow runs and have not completed a paid onboarding session (Source 1).',
 'sources': ['return_policy.md', 'company_faq.md']}